# CleanLearning.ipynb

[Gemini link](https://share.google/aimode/yjKHD5GeEXMqyxGYC)

#### References

[Run a Free AI Coding Assistant Locally in VS Code](https://pankajchejara.medium.com/run-a-free-ai-coding-assistant-locally-in-vs-code-3ab2ef3a36eb)

In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from cleanlab.classification import CleanLearning
import pandas as pd
from icecream import ic
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix
import duckdb
from IPython.display import HTML, display

In [2]:
training_dataset_csv = 'training_dataset_202611221400.csv' 
tree_silhouettes_gallery_path = 'tree_silhouettes_202611201700'  

## Cell 1: Data preparation

In [3]:
df = pd.read_csv(training_dataset_csv)
df

,tree_id,dead,h1,h2,h3,h4,h5,h6,h7
0,77,0,-0.524697,-1.482314,-2.328288,-2.914501,-5.633616,-3.723588,-5.756312
1,179,0,-0.058715,-0.151640,-0.348927,-0.381639,-0.746923,-0.458197,3.379063
2,198,0,0.008442,-0.017242,-0.533576,-0.607849,-1.178735,-0.620762,-2.727663
3,214,0,-0.242159,-0.589162,-1.049723,-1.081010,-2.146467,-1.393966,3.835318
4,218,0,0.278500,0.551319,-0.444942,-0.489195,-0.956292,-0.223162,-2.899226
...,...,...,...,...,...,...,...,...,...
495,26698,0,-0.348136,-0.820270,-1.167873,-1.270130,-2.489414,-1.687492,3.931697
496,26880,0,0.227771,0.445403,-0.206446,-0.225895,-0.442066,-0.003300,3.189195
497,26887,0,-0.248995,-0.593392,-0.592517,-0.668806,-1.299471,-0.966028,3.666268
498,26948,0,-0.674271,-2.903272,-3.169987,-4.325097,9.990137,-5.816504,-8.072671


In [4]:
# parse dataset into numpy arrays
X = df.drop(['tree_id', 'dead', 'h7'], axis=1).to_numpy()
# ic(X)
y = df[['dead']].to_numpy().flatten()
# ic(y)
meta = df[['tree_id']].to_numpy().flatten()
# ic(meta);

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=42)

In [6]:
# 1. Initialize RandomForestClassifier with your specific parameters
clf = RandomForestClassifier(    
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42,
    # n_estimators=50,
    # max_depth=12,
    # min_samples_split=2,
    # class_weight="balanced",
    # n_jobs=-1,
    # random_state=42
)

In [7]:
clf_before = clone(clf)
clf_before.fit(X_train, y_train)
probs_before = clf_before.predict_proba(X_test)[:, 1]
auc_before = roc_auc_score(y_test, probs_before)
preds_before = clf_before.predict(X_test)
cm_before = confusion_matrix(y_test, preds_before)

In [8]:
# 2. Wrap the configured model inside CleanLearning
cl = CleanLearning(
    clf=clf,
    cv_n_folds=5,  # Number of cross-validation folds for finding label issues
    seed=42,
    verbose=True
)

In [9]:
# 3. Fit on your dataset (automatically finds issues & retrains rf_model on clean data)
cl.fit(X_train, y_train)

probs_after = cl.predict_proba(X_test)[:, 1]
auc_after = roc_auc_score(y_test, probs_after)
preds_after = cl.predict(X_test)
cm_after = confusion_matrix(y_test, preds_after)

print('\nBefore pruning by CleanLearning:')
print(f"ROC AUC: {auc_before:.4f}")
print("Confusion Matrix:\n", cm_before)

print('\nAfter pruning by CleanLearning:')
print(f"ROC AUC:  {auc_after:.4f}")
print("Confusion Matrix:\n", cm_after)

Computing out of sample predicted probabilities via 5-fold cross validation. May take a while ...
Using predicted probabilities to identify label issues ...
Identified 20 examples with label issues.
Pruning 20 examples with label issues ...
Remaining clean data has 380 examples.
Assigning sample weights for final training based on estimated label quality.
Fitting final model on the clean data ...
Label issues stored in label_issues_df DataFrame accessible via: self.get_label_issues(). Call self.save_space() to delete this potentially large DataFrame attribute.

Before pruning by CleanLearning:
ROC AUC: 0.9783
Confusion Matrix:
 [[88  4]
 [ 0  8]]

After pruning by CleanLearning:
ROC AUC:  0.9715
Confusion Matrix:
 [[87  5]
 [ 0  8]]


In [10]:
issues_df = cl.get_label_issues()
duckdb.sql('SELECT * FROM issues_df WHERE is_label_issue ORDER BY label_quality;').df()

,is_label_issue,label_quality,given_label,predicted_label,sample_weight
0,True,0.000000,1,0,0.0
1,True,0.024736,1,0,0.0
2,True,0.037201,0,1,0.0
3,True,0.061677,0,1,0.0
4,True,0.062171,1,0,0.0
5,True,0.079262,0,1,0.0
6,True,0.086037,0,1,0.0
7,True,0.093663,1,0,0.0
8,True,0.102964,0,1,0.0
9,True,0.188294,0,1,0.0


# KEEP

In [11]:
from pathlib import Path
from IPython.display import display
import ipywidgets as widgets

classes = ["alive", "dead"]
tree_ids = [220, 272, 281]
image_paths = [Path(f"{tree_silhouettes_gallery_path}/{tree_id}.svg") for tree_id in tree_ids]

labels = {}
index = 0

# UI Elements
status_label = widgets.Label()
image_widget = widgets.Image(
    format="svg+xml", layout=widgets.Layout(max_width="200px", max_height="200px")
)


def show_next():
  if index < len(image_paths):
    current_path = image_paths[index]
    # label = 'alive'
    # status_label.value = f'Image {index + 1}/{len(image_paths)}: {current_path.name} label: {label}'
    status_label.value = f'Image {index + 1}/{len(image_paths)}: {current_path.name}'
    image_widget.value = current_path.read_bytes()
  else:
    status_label.value = "All SVG images labeled!"
    image_widget.layout.display = "none"


def on_button_click(b):
  global index
  if index < len(image_paths):
    labels[image_paths[index].name] = b.description
    index += 1
    show_next()


buttons = [widgets.Button(description=c) for c in classes]
for btn in buttons:
  btn.on_click(on_button_click)

# Display UI
display(widgets.VBox([widgets.HBox(buttons), status_label, image_widget]))
show_next()

In [14]:
ic(labels);

ic| labels: {'220.svg': 'alive', '272.svg': 'alive', '281.svg': 'dead'}
